In [1]:
import os
from database.db import Database
from database.db import Record
from database.ql import QLExecutor

In [2]:
os.system("rm -rf example.mydb")

0

In [3]:
db = Database("example.mydb")

In [4]:
db.open()

In [5]:
executor = QLExecutor(db)

In [6]:
text = """
CREATE TABLE users (
    id INTEGER,
    name STRING,
    age INTEGER,
    height INTEGER,
    INDEX (name),
    INDEX (age),
    INDEX (height),
    PRIMARY KEY (id)
);

INSERT INTO users (id, name, age, height) VALUES (1, 'Alice', 25, 165 + -(50 % 32));
INSERT INTO users (id, name, age, height) VALUES (2, 'Bob', 30, 175 - 50 % 32);
INSERT INTO users (id, name, age, height) VALUES (3, 'Cindy', 35, 185 - 50 % 32);
INSERT INTO users (id, name, age, height) VALUES (4, 'David', 40, 195 - 50 % 32);

SELECT id AS user_id, name, age, height FROM users INDEX BY name >= 'Bob' AND name <= 'Dob' FILTER age <= 40 LIMIT 10;
"""

In [7]:
recs = executor.execute(text)
recs

[Record(user_id=2, name=b'Bob', age=30, height=157),
 Record(user_id=3, name=b'Cindy', age=35, height=167),
 Record(user_id=4, name=b'David', age=40, height=177)]

In [8]:
print(Record.format_records(recs))

# 3 records found
‖ user_id    ‖ name       ‖ age        ‖ height    ‖
| 2          | Bob        | 30         | 157       |
----------------------------------------------------
| 3          | Cindy      | 35         | 167       |
----------------------------------------------------
| 4          | David      | 40         | 177       |
----------------------------------------------------



In [9]:
recs = executor.execute("""
UPDATE users SET age = 50 INDEX BY name >= 'Bob' AND name <= 'Bob';
SELECT id AS user_id, name, age, height FROM users INDEX BY name >= 'Bob' AND name <= 'Dob' FILTER age <= 40 LIMIT 10;
""")
print(Record.format_records(recs))

# 2 records found
‖ user_id    ‖ name       ‖ age        ‖ height    ‖
| 3          | Cindy      | 35         | 167       |
----------------------------------------------------
| 4          | David      | 40         | 177       |
----------------------------------------------------



In [10]:
recs = executor.execute("""
DELETE FROM users INDEX BY name >= 'David' AND name <= 'David';
SELECT id AS user_id, age, name FROM users INDEX BY id > 1 FILTER age <= 40 LIMIT 10;
""")
print(Record.format_records(recs))

# 1 record found
‖ user_id    ‖ age        ‖ name      ‖
| 3          | 35         | Cindy     |
---------------------------------------



In [11]:
recs = executor.execute("""
DELETE FROM users INDEX BY name >= 'David' AND name <= 'David';
SELECT id AS user_id, age, name FROM users INDEX BY id > 1 FILTER age <= 40 LIMIT 10;
""")
print(Record.format_records(recs))

# 1 record found
‖ user_id    ‖ age        ‖ name      ‖
| 3          | 35         | Cindy     |
---------------------------------------



In [12]:
recs = executor.execute("""
DROP TABLE users;
""")

In [13]:
try:
  recs = executor.execute("""
  SELECT id AS user_id, age, name FROM users INDEX BY id > 1 FILTER age <= 40 LIMIT 10;
  """)
  print(Record.format_records(recs))
except KeyError as e:
  print(e)

'Table not found: users'


In [14]:
db.kv.tree._get(db.kv.tree.root)

BNode: LEAF
  Children: 0, 0
  Key-Value:
    b'': b'' 
    b'\x01\x00\x00\x00next_prefix\x00': b'\x07\x01\x01\x01\x01\x01\x01\x00' 

In [15]:
db.close()